# 은평구 대상 스테이션 데이터 정리

이 노트는 분석 대상 스테이션만 추려서 시간 기준을 맞추고, 기상 데이터를 결합한 뒤 저장하는 전처리용 노트이다.

정리 순서는 다음과 같다.
1. 대상 스테이션 필터링
2. 도착시간 데이터를 출발시간 기준으로 정규화
3. 날짜/시간/이용시간 컬럼 타입 정리
4. 시간대별 기상 데이터 결합
5. parquet 파일 저장

In [1]:
import pandas as pd

## 1. 대상 스테이션 추출과 시간 기준 통일

먼저 분석에 사용할 스테이션 ID 목록을 고정한 뒤, 시작 대여소 또는 종료 대여소가 해당 목록에 포함된 행만 남긴다.

그다음 `도착시간` 기준으로 집계된 행은 `전체_이용_분`을 이용해 출발 시각으로 되돌려서, 전체 데이터를 `출발시간` 기준으로 맞춘다.

In [ ]:
station_ids = [
    'ST-481', # 상현
    'ST-2425', # 다원
    'ST-1331', # 찬솔
    'ST-454', # 신영
    'ST-453', # 혜전
    'ST-1482', # 광태
]

def filter_station_rows(df):
    return df[
        df['시작_대여소_ID'].isin(station_ids)
        | df['종료_대여소_ID'].isin(station_ids)
    ].copy()

def drop_internal_arrival_duplicates(df):
    internal_trip_mask = (
        df['시작_대여소_ID'].isin(station_ids)
        & df['종료_대여소_ID'].isin(station_ids)
    )
    duplicate_mask = internal_trip_mask & df['집계_기준'].eq('도착시간')
    removed_count = int(duplicate_mask.sum())
    return df.loc[~duplicate_mask].copy(), removed_count

def normalize_to_departure_time(df):
    df = df.copy()
    date_str = df['기준_날짜'].astype(str).str.replace('.0', '', regex=False).str.zfill(8)
    time_str = (
        pd.to_numeric(df['기준_시간대'], errors='coerce')
        .fillna(0)
        .astype(int)
        .astype(str)
        .str.zfill(4)
    )
    base_dt = pd.to_datetime(date_str + time_str, format='%Y%m%d%H%M', errors='coerce')

    arrival_mask = df['집계_기준'].eq('도착시간')
    use_minutes = pd.to_numeric(df['전체_이용_분'], errors='coerce').fillna(0)
    base_dt.loc[arrival_mask] = base_dt.loc[arrival_mask] - pd.to_timedelta(
        use_minutes.loc[arrival_mask], unit='m'
    )

    df['기준_날짜'] = base_dt.dt.strftime('%Y%m%d')
    df['기준_시간대'] = base_dt.dt.strftime('%H%M').astype(float)
    df['집계_기준'] = '출발시간'
    return df

df1 = pd.read_csv('../../../Data/정류장정보_시간대별_합친것.csv')
df2 = pd.read_parquet('../../../Data/2025_data.parquet')

df1 = filter_station_rows(df1)
df2 = filter_station_rows(df2)

df1, df1_removed_duplicates = drop_internal_arrival_duplicates(df1)
df2, df2_removed_duplicates = drop_internal_arrival_duplicates(df2)

df1 = normalize_to_departure_time(df1)
df2 = normalize_to_departure_time(df2)

drop_cols = ['시작_대여소명', '종료_대여소명']
df1 = df1.drop(columns=drop_cols, errors='ignore')
df2 = df2.drop(columns=drop_cols, errors='ignore')

dedupe_summary = pd.DataFrame(
    [
        {'구분': '2024', '삭제된_중복_건수': df1_removed_duplicates, '남은_행수': len(df1)},
        {'구분': '2025', '삭제된_중복_건수': df2_removed_duplicates, '남은_행수': len(df2)},
    ]
)

dedupe_summary

## 2. 필터링 결과 확인

전처리 직후 2024년 데이터와 2025년 데이터의 샘플을 확인한다. 여기서는 대상 스테이션 필터링과 출발시간 기준 변환이 적용된 상태를 점검한다.

In [37]:
df1.head()

,기준_날짜,집계_기준,기준_시간대,시작_대여소_ID,종료_대여소_ID,전체_건수,전체_이용_분,전체_이용_거리,대여소_ID,주소1,주소2,위도,경도
0,20240101,출발시간,1735.0,ST-479,ST-462,2.0,151.0,21873.0,ST-479,서울특별시 은평구 역촌동 45-34,역촌파출소,37.604736,126.915337
2,20240101,출발시간,1654.0,ST-460,ST-2425,1.0,11.0,550.0,ST-460,서울특별시 은평구 응암동 604-5,응암오거리,37.589661,126.916946
3,20240101,출발시간,1650.0,ST-461,ST-2425,1.0,15.0,1582.0,ST-461,서울특별시 은평구 응암동 90-15,이마트 은평점,37.600700,126.920128
4,20240101,출발시간,1643.0,ST-479,ST-479,1.0,22.0,2526.0,ST-479,서울특별시 은평구 역촌동 45-34,역촌파출소,37.604736,126.915337
6,20240101,출발시간,1710.0,ST-2264,ST-479,1.0,2.0,445.0,ST-2264,서울특별시 은평구 연서로 9 센타폴리스,NaN,37.599968,126.915726


In [38]:
df2.head()

,기준_날짜,집계_기준,기준_시간대,시작_대여소_ID,종료_대여소_ID,전체_건수,전체_이용_분,전체_이용_거리
0,20250109,출발시간,900.0,ST-2775,ST-462,1,9,1379
1,20250109,출발시간,1905.0,ST-462,ST-2775,1,10,1368
2,20250109,출발시간,1625.0,ST-2782,ST-481,1,8,1281
4,20250109,출발시간,1610.0,ST-2261,ST-3126,1,8,1060
5,20250109,출발시간,1705.0,ST-32,ST-462,1,48,5946


## 3. 컬럼 정제와 타입 변환

- `기준_날짜`는 datetime 타입으로 변환한다.
- `기준_시간대`는 시간 단위 `시간대` 컬럼으로 바꾼다.
- `전체_이용_분`은 timedelta 형식으로 정리한다.
- 불필요한 위치/주소 컬럼은 필요에 따라 제거한다.

In [ ]:
def refine_trip_columns(df, drop_extra_cols=None):
    result = df.copy()

    date_str = result['기준_날짜'].astype(str).str.replace('.0', '', regex=False).str.zfill(8)
    time_str = (
        pd.to_numeric(result['기준_시간대'], errors='coerce')
        .fillna(0)
        .astype(int)
        .astype(str)
        .str.zfill(4)
    )
    use_minutes = pd.to_numeric(result['전체_이용_분'], errors='coerce').fillna(0)

    result['기준_날짜'] = pd.to_datetime(date_str, format='%Y%m%d', errors='coerce')
    result['시간대'] = pd.to_datetime(time_str, format='%H%M', errors='coerce').dt.hour
    result['전체_이용_분'] = pd.to_timedelta(use_minutes, unit='m')
    result = result.drop(columns=['기준_시간대'], errors='ignore')

    if drop_extra_cols:
        result = result.drop(columns=drop_extra_cols, errors='ignore')

    return result

df1_refined = refine_trip_columns(
    df1,
    drop_extra_cols=['대여소_ID', '주소1', '주소2', '위도', '경도']
)
df2_refined = refine_trip_columns(df2)

df1_refined = df1_refined[
    df1_refined['기준_날짜'].between(pd.Timestamp('2024-01-01'), pd.Timestamp('2024-12-31'))
].copy()
df2_refined = df2_refined[
    df2_refined['기준_날짜'].between(pd.Timestamp('2025-01-09'), pd.Timestamp('2025-12-31'))
].copy()

date_range_summary = pd.DataFrame(
    [
        {
            '구분': '2024_refined',
            '시작일': df1_refined['기준_날짜'].min().date(),
            '종료일': df1_refined['기준_날짜'].max().date(),
            '삭제된_중복_건수': df1_removed_duplicates,
            '행수': len(df1_refined),
        },
        {
            '구분': '2025_refined',
            '시작일': df2_refined['기준_날짜'].min().date(),
            '종료일': df2_refined['기준_날짜'].max().date(),
            '삭제된_중복_건수': df2_removed_duplicates,
            '행수': len(df2_refined),
        },
    ]
)

date_range_summary


In [40]:
df2_refined

,기준_날짜,집계_기준,시작_대여소_ID,종료_대여소_ID,전체_건수,전체_이용_분,전체_이용_거리,시간대
0,2025-01-09,출발시간,ST-2775,ST-462,1,0 days 00:09:00,1379,9
1,2025-01-09,출발시간,ST-462,ST-2775,1,0 days 00:10:00,1368,19
2,2025-01-09,출발시간,ST-2782,ST-481,1,0 days 00:08:00,1281,16
4,2025-01-09,출발시간,ST-2261,ST-3126,1,0 days 00:08:00,1060,16
5,2025-01-09,출발시간,ST-32,ST-462,1,0 days 00:48:00,5946,17
...,...,...,...,...,...,...,...,...
1853017,2025-12-31,출발시간,ST-1035,ST-1038,1,0 days 00:00:00,0,16
1853018,2025-12-31,출발시간,ST-471,ST-2261,1,0 days 01:04:00,2155,12
1853021,2025-12-31,출발시간,ST-2261,ST-481,1,0 days 00:08:00,2025,8
1853022,2025-12-31,출발시간,ST-226,ST-1034,1,0 days 00:04:00,634,6


## 4. 시간대별 기상 데이터 결합

Open-Meteo 아카이브 API에서 서울 좌표 기준 시간별 날씨를 불러와 각 행의 `기준_날짜 + 시간대`와 맞춰 병합한다.

추가되는 주요 변수는 다음과 같다.
- `온도`
- `습도`
- `불쾌지수`
- `강수량`
- `적설량`

In [41]:
import requests

SEOUL_LAT = 37.607853
SEOUL_LON = 126.916525

def fetch_hourly_weather(start_date, end_date, latitude=SEOUL_LAT, longitude=SEOUL_LON):
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'start_date': start_date,
        'end_date': end_date,
        'hourly': 'temperature_2m,precipitation,snowfall,relative_humidity_2m',
        'timezone': 'Asia/Seoul',
    }
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()

    hourly = response.json()['hourly']
    weather_df = pd.DataFrame(hourly).rename(columns={
        'time': '기준_일시',
        'temperature_2m': '온도',
        'relative_humidity_2m': '습도',
        'precipitation': '강수량',
        'snowfall': '적설량',
    })
    weather_df['기준_일시'] = pd.to_datetime(weather_df['기준_일시'])
    weather_df['불쾌지수'] = (
        0.81 * weather_df['온도']
        + 0.01 * weather_df['습도'] * (0.99 * weather_df['온도'] - 14.3)
        + 46.3
    )
    return weather_df[['기준_일시', '온도', '습도', '불쾌지수', '강수량', '적설량']]

def merge_weather_by_hour(df, weather_df):
    result = df.copy()
    result['기준_일시'] = result['기준_날짜'] + pd.to_timedelta(result['시간대'], unit='h')
    result = result.merge(weather_df, on='기준_일시', how='left')
    result = result.drop(columns=['기준_일시'], errors='ignore')

    front_cols = ['기준_날짜', '시간대']
    other_cols = [col for col in result.columns if col not in front_cols]
    return result[front_cols + other_cols]

weather_2024 = fetch_hourly_weather('2024-01-01', '2024-12-31')
weather_2025 = fetch_hourly_weather('2025-01-01', '2025-12-31')

df1_weather = merge_weather_by_hour(df1_refined, weather_2024)
df2_weather = merge_weather_by_hour(df2_refined, weather_2025)

df1_weather.head()

,기준_날짜,시간대,집계_기준,시작_대여소_ID,종료_대여소_ID,전체_건수,전체_이용_분,전체_이용_거리,온도,습도,불쾌지수,강수량,적설량
0,2024-01-01,17,출발시간,ST-479,ST-462,2.0,0 days 02:31:00,21873.0,4.3,78,41.94946,0.0,0.0
1,2024-01-01,16,출발시간,ST-460,ST-2425,1.0,0 days 00:11:00,550.0,6.1,65,45.87135,0.0,0.0
2,2024-01-01,16,출발시간,ST-461,ST-2425,1.0,0 days 00:15:00,1582.0,6.1,65,45.87135,0.0,0.0
3,2024-01-01,16,출발시간,ST-479,ST-479,1.0,0 days 00:22:00,2526.0,6.1,65,45.87135,0.0,0.0
4,2024-01-01,17,출발시간,ST-2264,ST-479,1.0,0 days 00:02:00,445.0,4.3,78,41.94946,0.0,0.0


In [42]:
df2_weather.head()

,기준_날짜,시간대,집계_기준,시작_대여소_ID,종료_대여소_ID,전체_건수,전체_이용_분,전체_이용_거리,온도,습도,불쾌지수,강수량,적설량
0,2025-01-09,9,출발시간,ST-2775,ST-462,1,0 days 00:09:00,1379,-12.1,34,27.56414,0.0,0.0
1,2025-01-09,19,출발시간,ST-462,ST-2775,1,0 days 00:10:00,1368,-12.1,30,28.61530,0.0,0.0
2,2025-01-09,16,출발시간,ST-2782,ST-481,1,0 days 00:08:00,1281,-10.0,22,32.87600,0.0,0.0
3,2025-01-09,16,출발시간,ST-2261,ST-3126,1,0 days 00:08:00,1060,-10.0,22,32.87600,0.0,0.0
4,2025-01-09,17,출발시간,ST-32,ST-462,1,0 days 00:48:00,5946,-11.1,25,30.98675,0.0,0.0


## 5. 전처리 결과 저장

최종적으로 기상 정보까지 결합된 2024년, 2025년 데이터를 `../../Data/sort_data` 폴더에 parquet 형식으로 저장한다.

In [ ]:
from pathlib import Path

output_dir = Path('../../Data/sort_data')
output_dir.mkdir(parents=True, exist_ok=True)

save_summary = pd.DataFrame(
    [
        {
            '파일': '2024_data.parquet',
            '시작일': df1_weather['기준_날짜'].min().date(),
            '종료일': df1_weather['기준_날짜'].max().date(),
            '삭제된_중복_건수': df1_removed_duplicates,
            '행수': len(df1_weather),
        },
        {
            '파일': '2025_data.parquet',
            '시작일': df2_weather['기준_날짜'].min().date(),
            '종료일': df2_weather['기준_날짜'].max().date(),
            '삭제된_중복_건수': df2_removed_duplicates,
            '행수': len(df2_weather),
        },
    ]
)

df1_weather.to_parquet(output_dir / '2024_data.parquet', index=False)
df2_weather.to_parquet(output_dir / '2025_data.parquet', index=False)

print(output_dir / '2024_data.parquet')
print(output_dir / '2025_data.parquet')
save_summary
